In [4]:
!pip install transformers datasets accelerate evaluate

In [5]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
from torch.utils.data import Dataset

In [6]:
news_df = pd.read_csv("../data/processed/final_news.csv")

In [7]:
news_df.head()

,title,text,subject,date,label,text_length,content
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,"A school librarian in Cambridge, Massachusetts...",left-news,"Sep 28, 2017",0,10062,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...
1,Kenya opposition leader calls for calm in slum...,NAIROBI (Reuters) - Kenyan opposition leader R...,worldnews,"October 29, 2017",1,2808,Kenya opposition leader calls for calm in slum...
2,Egypt rejects U.S. decision to move its embass...,CAIRO (Reuters) - Egypt rejected the U.S. deci...,worldnews,"December 6, 2017",1,232,Egypt rejects U.S. decision to move its embass...
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,After a recent speech given by Minister Louis ...,left-news,"May 8, 2015",0,610,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,Tensions ran high outside of a campaign rally ...,News,"March 11, 2016",0,2658,Trump Rally Nearly Turns Into A Full-Blown Rac...


In [8]:
news_df.shape

(44058, 7)

In [9]:
news_df.isnull().sum()

title          0
text           0
subject        0
date           0
label          0
text_length    0
content        0
dtype: int64

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    news_df["content"],
    news_df["label"],
    test_size=0.2,
    random_state=42,
    stratify=news_df["label"]
)

In [11]:
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 35246
Testing Samples: 8812


In [12]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [13]:
sample_text = "Artificial Intelligence is changing the world."

tokens = tokenizer.tokenize(sample_text)

print(tokens)

['artificial', 'intelligence', 'is', 'changing', 'the', 'world', '.']


In [14]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(token_ids)

[7976, 4454, 2003, 5278, 1996, 2088, 1012]


In [15]:
encoded = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=20,
    return_tensors="pt"
)

In [16]:
print(encoded["input_ids"])

tensor([[ 101, 7976, 4454, 2003, 5278, 1996, 2088, 1012,  102,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]])


In [17]:
print(encoded["attention_mask"])

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


In [18]:
print(tokenizer.decode(encoded["input_ids"][0]))

[CLS] artificial intelligence is changing the world. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [19]:
train_texts = X_train.tolist()
test_texts = X_test.tolist()

train_labels = y_train.tolist()
test_labels = y_test.tolist()

In [20]:
print(type(train_texts))
print(type(train_labels))

<class 'list'>
<class 'list'>


In [21]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=512
)

In [22]:
test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=512
)

In [23]:
class NewsDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {}

        for key, val in self.encodings.items():
            item[key] = torch.tensor(val[idx])

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [24]:
train_dataset = NewsDataset(train_encodings, train_labels)

test_dataset = NewsDataset(test_encodings, test_labels)

In [25]:
print(len(train_dataset))
print(len(test_dataset))

35246
8812


In [26]:
train_dataset[0]

{'input_ids': tensor([  101,  1520,  2025,  1011, 16939,  1521,  8398,  5470,  3632,  2006,
          2847,  1011,  2146, 10474,  2743,  2102,  2055,  1520,  9152,  5620,
          1521,  1998,  1520,  8112, 19093,  1521,  1006,  1056, 28394,  3215,
          1007,  2065,  2017,  2412,  2342,  2000,  2113,  3599,  2129,  4487,
         11365,  3512,  6221,  8398,  2038,  2042,  1010,  2065,  2017,  2412,
          2342,  1037,  3819,  2742,  1010,  2017,  2123,  1056,  2342,  2000,
          2298,  2172,  2582,  2084,  6221,  8398,  1055, 10474, 17060,  1012,
          2096,  2009,  2003,  1037,  8292,  4757, 23270,  1997, 16939, 13044,
          2006,  1037,  2204,  2154,  1010,  2823,  2010,  4599,  3233,  2041,
          1998,  2028,  1997,  2068,  4240,  2004,  1037,  3819, 14764,  1997,
          2054,  1010,  3599,  2057,  2024,  2157,  2075,  2114,  1012,  2006,
          4465,  1010,  8398, 23678,  2098,  2055,  2010,  8599,  1999,  1037,
          7143,  3947,  2000, 15886,  2